# Phantom Ink Question Generator

AI 驅動的「靈媒遊戲」題目生成器

## 流程
1. **出題** — AI 扮演出題老師，產生七道由難到易的問答
2. **驗題** — AI 扮演驗題老師，檢查題目品質
3. **模擬** — AI 扮演玩家，逐題模擬猜測過程
4. **注音轉換** — 自動將回答轉換為注音符號

## 第一步：安裝相依套件

In [ ]:
# @title 安裝相依套件
!pip install pypinyin openai pydantic -q
# Hugging Face 後端用（Colab 已有 torch）
!pip install transformers accelerate bitsandbytes -q

## 第二步：設定 API Key

支援 OpenAI 或 Gemini（透過 OpenAI 相容 API）

- OpenAI：`sk-...`
- Gemini：透過 `https://generativelanguage.googleapis.com/v1beta/openai/`

In [ ]:
# @title API / 後端設定

BACKEND_TYPE = "openai"  # @param ["openai", "huggingface"]
API_KEY = ""  # @param {type: "string"}
MODEL_NAME = "gpt-4o"  # @param {type: "string"}

# Hugging Face 模型（僅 huggingface 後端使用）
HF_MODEL = "Qwen/Qwen2.5-7B-Instruct"  # @param {type: "string"}

import os

os.environ["BACKEND_TYPE"] = BACKEND_TYPE

if BACKEND_TYPE == "openai":
    os.environ["OPENAI_API_KEY"] = API_KEY
    os.environ["LLM_MODEL"] = MODEL_NAME
    print(f"✅ 後端：OpenAI / {MODEL_NAME}")
else:
    os.environ["HF_MODEL"] = HF_MODEL
    print(f"✅ 後端：Hugging Face / {HF_MODEL}")
    print(f"   首次使用會下載模型（約 4GB），請耐心等待...")

## 第三步：從 GitHub 下載模組

Colab 無法直接引用本機 .py 檔，這裡直接從 GitHub Raw 抓取所有必要模組。

In [ ]:
# @title 下載模組
import os
import sys

GITHUB_RAW = "https://raw.githubusercontent.com/csongs/Phantom-Ink-Question/master"

FILES = ["backends.py", "models.py", "prompts.py", "bopomofo.py", "generator.py", "utils.py"]

for f in FILES:
    url = f"{GITHUB_RAW}/{f}"
    !wget -q -O {f} {url} || curl -s -o {f} {url}
    if os.path.exists(f):
        print(f"  ✓ {f}")
    else:
        print(f"  ✗ {f} 下載失敗（可忽略非必要檔案）")

# 匯入模組
from generator import PhantomInkGenerator
from models import QuestionSetWithMeta, QuestionItem
from bopomofo import to_bopomofo, to_bopomofo_cells, reveal_bopomofo, count_bopomofo_cells
from utils import save_question_set, load_question_set, export_to_colab_json, print_question_set

print("\n✅ 全部模組載入完成")

## 第四步：單題生成

輸入一個謎底，自動完成出題 → 驗題 → 模擬玩家

In [ ]:
# @title 單題生成

ANSWER = "鋼琴"  # @param {type: "string"}
SKIP_REVIEW = False  # @param {type: "boolean"}
SKIP_SIMULATION = False  # @param {type: "boolean"}
MAX_RETRIES = 3  # @param {type: "integer"}

backend_type = os.environ.get("BACKEND_TYPE", "openai")

if backend_type == "huggingface":
    gen = PhantomInkGenerator(
        backend="huggingface",
        hf_model=os.environ.get("HF_MODEL", "Qwen/Qwen2.5-7B-Instruct"),
        max_retries=MAX_RETRIES,
    )
else:
    gen = PhantomInkGenerator(
        backend="openai",
        api_key=os.environ.get("OPENAI_API_KEY", ""),
        base_url=os.environ.get("OPENAI_BASE_URL"),
        model=os.environ.get("LLM_MODEL", "gpt-4o"),
        max_retries=MAX_RETRIES,
    )

result = gen.generate(
    answer=ANSWER,
    skip_review=SKIP_REVIEW,
    skip_simulation=SKIP_SIMULATION,
    verbose=True,
)

print_question_set(result)

## 第五步：批次生成

一次生成多個謎底的題組，適合大量建立題庫

In [ ]:
# @title 批次生成

# 輸入謎底清單（一行一個）
ANSWERS_TEXT = """鋼琴
颱風
相機
腳踏車
冰箱"""  # @param {type: "string"}

answers = [a.strip() for a in ANSWERS_TEXT.split("\n") if a.strip()]

backend_type = os.environ.get("BACKEND_TYPE", "openai")

if backend_type == "huggingface":
    gen = PhantomInkGenerator(
        backend="huggingface",
        hf_model=os.environ.get("HF_MODEL", "Qwen/Qwen2.5-7B-Instruct"),
        max_retries=3,
    )
else:
    gen = PhantomInkGenerator(
        backend="openai",
        api_key=os.environ.get("OPENAI_API_KEY", ""),
        base_url=os.environ.get("OPENAI_BASE_URL"),
        model=os.environ.get("LLM_MODEL", "gpt-4o"),
        max_retries=3,
    )

results = gen.generate_batch(answers)

# 顯示結果摘要
print("\n" + "=" * 60)
print("📊 批次生成結果")
print("=" * 60)
for r in results:
    status = "✅" if r.review and r.review.passed else "⚠️"
    score = r.review.score if r.review else "N/A"
    sim_info = f"第{r.simulation.guess_round}題猜出" if r.simulation else "無模擬"
    print(f"  {status} {r.answer}: {score}分 | {sim_info}")
    for q in r.questions:
        bpmf = to_bopomofo(q.reply)
        print(f"      ├ {q.question}")
        print(f"      └ {q.reply} ({bpmf})")
    print()

## 第六步：匯出題庫

將生成的題組儲存為 JSON 或 CSV 格式

In [ ]:
# @title 匯出結果

EXPORT_FORMAT = "json"  # @param ["json", "csv"]

import datetime
import csv

timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")

if EXPORT_FORMAT == "json":
    for r in results:
        filename = f"phantom_ink_{r.answer}_{timestamp}.json"
        save_question_set(r, filename, format="json")
        export_to_colab_json(r, f"game_{r.answer}_{timestamp}.json")
        print(f"\u2705 \u5df2\u5f59\u51fa\uff1a{filename}")
else:
    filename = f"phantom_ink_batch_{timestamp}.csv"
    with open(filename, "w", encoding="utf-8-sig", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["answer", "question_number", "question", "reply"])
        for r in results:
            for i, q in enumerate(r.questions):
                writer.writerow([r.answer, i + 1, q.question, q.reply])
    print(f"\u2705 \u5df2\u5f59\u51fa\uff1a{filename}")

# Colab 觸發下載到本機
try:
    from google.colab import files
    files.download(filename)
    print("\U0001f4e5 \u5df2\u89f8\u767c\u4e0b\u8f09")
except:
    pass

## 第七步：注音墨水預覽

預覽每個回答的注音逐格顯示效果

In [ ]:
# @title 注音墨水預覽

if 'result' in locals() and result.questions:
    print(f"\n\U0001f3af \u8b0e\u5e95\uff1a{result.answer}")
    print("=" * 50)

    for i, q in enumerate(result.questions):
        cells = to_bopomofo_cells(q.reply)
        total = len(cells)
        print(f"\nQ{i+1}. {q.question}")
        print(f"\u56de\u7b54\uff1a{q.reply}")
        print(f"\u6ce8\u97f3\uff1a{' '.join(cells)}")
        print(f"\u683c\u6578\uff1a{total}")

        print("\u63ed\u9732\u904e\u7a0b\uff1a")
        for step in range(1, total + 1):
            revealed = reveal_bopomofo(q.reply, step)
            pct = int(step / total * 100)
            bar = "\u2588" * (pct // 10) + "\u2591" * (10 - pct // 10)
            print(f"  [{bar}] {revealed}")

    print("\n\u2705 \u9810\u89bd\u5b8c\u6210")
else:
    print("\u26a0\ufe0f \u8acb\u5148\u57f7\u884c\u300c\u55ae\u984c\u751f\u6210\u300d\u6216\u300c\u6279\u6b21\u751f\u6210\u300d")

## 除錯工具

直接測試注音轉換功能

In [ ]:
# @title 注音轉換測試

TEST_WORDS = ["樂器行", "演奏廳", "鍵盤樂器", "木頭", "颱風", "腳踏車"]

print("測試注音轉換\n")
print(f"{'詞語':<10} {'注音':<30} {'格數':<6}")
print("-" * 50)

for word in TEST_WORDS:
    bpmf = to_bopomofo(word)
    cells = count_bopomofo_cells(word)
    print(f"{word:<10} {bpmf:<30} {cells:<6}")